---
authors:
  - edesz
date: 2026-05-07
---

# Estimate Cohort Size With Predicted Net Savings

## About

In this step, we will use the best ML model's prediction probabilities and an estimate of the (net) savings from targeting at-risk customers to identify this cohort for two assumed budget scenarios.

### Use-Case Context

Same as in [the previous step using ROI](./08_estimate_cohort_size_using_roi.ipynb) for but net savings.

### Relevant Assumptions

Same as in the previous step

:::{note}
### Outputs

Same as in the previous step.

The filenames will contain the word `savings`.
:::

## Python Imports

The required Python modules are imported below

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import altair as alt
import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from great_tables import GT, loc, md, style
from IPython.display import Markdown

The plotting settings for `altair` are below

In [ ]:
_ = alt.renderers.set_embed_options(actions=False)

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

Import the required custom modules

In [ ]:
import cc_churn.costs as costs
import cc_churn.costs_savings as costs_sv
import cc_churn.viz_altair as vzu
import r2.io_utils as r2io
from utils.df_utils import show_df
from utils.display_utils import pygments_highlight

## User Inputs

Below we define variables that will be used later

In [ ]:
# columns to load
columns = [
    "clientnum",
    "card_category",
    "total_revolv_bal",
    "total_trans_amt",
    "model_name",
    "y_pred_proba",
    "y_pred",
    # "best_decision_threshold",
    "is_churned",
]

# costs
# # revenue from transactions (bank earns #% of transaction volume)
interchange_rate = 0.02
# # revenue from revolving balance (~20% interest)
apr = 0.18
# # fee revenue from credit card exposure (modeled from card type)
card_fees = {"Blue": 0, "Silver": 50, "Gold": 100, "Platinum": 200}
tenure_years = 3
discount = 0.9
# # percentage of churners who can be convinced to stay (i.e. success rate
# # of saving a churning customer)
success_rate = 0.40
# # cost of intervention to get a single customer to not churn (discounts,
# # call center time, retention offers, etc.)
intervention_cost = 50
# # maximum number of customers that can be targeted based on client's budget
num_customers_max = 400

# predictions prefix
# # folder containing predictions
prefix = "cloud-run"
# # prefix of filename with predictions
r2_key_pred = "all_predictions__"

We now use environment variables to define an authenticated `boto3` R2 client and define the CLV multiplier [as per the scope](../references/scope/02_costs.md#clv-multiplier)

In [ ]:
reports_dir = PROJ_ROOT / "reports"
figures_dir = reports_dir / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

# costs
multiplier = (1 - discount**tenure_years) / (1 - discount)

## Load Data with Predictions

Load predictions for all available customers

In [ ]:
df_all_pred = r2io.pandas_read_latest_parquet_r2(
    s3_client,
    bucket_name,
    f"{prefix}/",
    r2_key_pred,
    ".parquet.gzip",
    columns,
).astype({"card_category": "category", "model_name": "category"})

Extract name of best ML model from model predictions

In [ ]:
best_model_name = df_all_pred["model_name"].head(1).squeeze()

## Estimate Net Savings

Calculate the predicted and true net savings from targeting all customers predicted to churn (`y_pred == 1`)

In [ ]:
df_business_metrics = (
    df_all_pred.query("y_pred == 1")
    .pipe(
        lambda df: costs.calc_predicted_savings(
            df,
            interchange_rate=interchange_rate,
            apr=apr,
            card_fees=card_fees,
            multiplier=multiplier,
            success_rate=success_rate,
            intervention_cost=intervention_cost,
        )
    )
    .assign(
        true_savings=lambda df: np.vectorize(costs.calc_true_savings)(
            pred=df["y_pred"],
            true=df["is_churned"],
            success_rate=success_rate,
            clv=df["clv"],
            intervention_cost=intervention_cost,
        )
    )
)

:::{hint}
The predicted net savings is estimated using the `calc_predicted_savings()` function. This function returns two columns that will be used later: `clv` (predicted CLV) and expected savings (`expected_savings`).

The true net savings is estimated using the `calc_true_savings()` function. This value is returned for all customers in the `true_savings` column.
:::

Next, we'll bin these at-risk customers based on

1. predicted probability to churn (`y_pred_proba`)
   - this creates risk-level bins using predicted churn probability tiers
2. our estimate of the predicted CLV (`clv`)
   - This creates customer value (revenue) bins and is created using quantile-based revenue tiers. Value tiers are based on quantile-based CLV segmentation. This approach, often called [ABC Analysis (Value Tiering)](https://www.dinmo.com/customer-segmentation/abc-analysis/), isolates the highest-contributing *VIP* customers (e.g. the top decile or upper quantile Platinum tier) who justify the most aggressive retention efforts. Using quantiles ensures that the top tier represents a truly elite customer group rather than simply customers above an arbitrary average revenue threshold.

In [ ]:
df_business_metrics = costs_sv.get_buckets(
    df_business_metrics,
    bins_risk_values=[0.72, 0.81, 0.9, 1.0],
    bins_risk_labels=["Low", "Medium", "High"],
    bins_clv_tier_values=[0, 0.4, 0.7, 0.9, 1.0],
    bins_clv_tier_labels=["Bronze", "Silver", "Gold", "Platinum"],
)

## Estimate Cohort Size - Optimal Number of Customers to Target (`N`)

### Create Lookup Tool Based on Risk and Value Segments

We'll now get characteristics of each combination of risk and value bins created above

1. total (net) predicted savings per bin
   - this will give the total predicted savings per bin, realized by acting on the customers predicted to be at risk of canceling their credit card services
   - this is stored in the `expected_savings` column
2. total (net) true savings per bin
   - this will give the total true savings per bin, realized by acting on the customers who did churn
   - this is stored in the `true_savings` column
3. number of customers per bin
   - this is stored in the `num_customers` column
4. average predicted probability of churn per bin
   - this is stored in the `y_pred_proba` column
5. average CLV per bin
   - this is stored in the `clv` column
6. total targeting cost (`total_intervention_cost`) per bin
   - this is the product of the number of customers and our assumed targeting (intervention) cost per customer ($50)
   - this is stored in the `total_intervention_cost` column
7. expected savings per customer
   - this is a better metric than expected savings since each combination of risk and value bins contains a different number of customers
   - this is stored in the `expected_savings_per_customer` column
8. expected savings error per bin
   - percent difference between true and predicted savings per bin, relative to true savings
   - this is stored in the `savings_error_pct` column
9. recommended targeting strategy per bin
   - this is stored in the `save_offer` column

Below, we extract the nine characteristics from above

In [ ]:
# | label: overall-segment-attributes
df_summary_per_clv_tier_risk_level = costs_sv.summarize_campaign_mix(
    df_business_metrics,
    "expected_savings_per_customer",
    intervention_cost,
    False,
    False,
)

The output is shown below

In [ ]:
gt = (
    GT(df_summary_per_clv_tier_risk_level)
    .tab_header(md("**Strategy Lookup From Risk and Value**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["risk_level"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["value_tier"]),
    )
    .tab_style(
        style=[
            style.fill(color="teal"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["expected_savings_per_customer"]),
    )
    .tab_style(
        style=[
            style.fill(color="darkred"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["save_offer"]),
    )
    .fmt_number(
        columns=[
            "true_savings",
            "expected_savings",
            "y_pred_proba",
            "clv",
            "expected_savings_per_customer",
            "savings_error_pct",
        ],
        decimals=1,
    )
)
gt

**Notes**

1. If `expected_savings` is less than `true_savings` then the percent error in the expected savings (`savings_error_pct`) is negative.

**Observations**

1. For the *Bronze / High Risk* group, the `expected_savings` is negative. This means it costs more to try and save these customers than they are worth. The same is true for the *Bronze* tier customers at the other risk levels (Low and Medium). So, the client to avoid manual outreach for *Bronze* tier customers entirely.
2. The *High Risk* category has the most customers (`num_customers`). However, the most efficient way to utilize the client's budget is to target the individuals with the highest *expected_savings_per_customer*. The
   - *Platinum / High Risk* group is the most *profitable* to save, and returns approximately 500 dollars per customer
   - (savings-tradeoff)=
     *Platinum / Medium Risk* (at 465 dollars per capita) and *Platinum / Low Risk* (at 400 dollars per capita) groups are actually a better use of resources than the *Gold / High Risk* group (which comes in at 355 dollars per capita). So, although the latter group returns a higher predicted savings, as seen from `expected_savings` in the table, the former two groups, even though they are at a lower risk level, are actually more efficient.
3. The Platinum and Gold customers within the High-Risk segment represent *must-retain* cohorts because their expected lifetime value strongly exceeds the cost of intervention. In practice, losing a single Platinum customer may have the same financial impact as losing many Bronze customers combined. As a result, the retention strategy intentionally allocated to these customers should be the most personalized and resource-intensive interventions such as executive outreach, fee waivers, and loyalty rewards.

Based on the above observations, below is the rationale for the recommended `save_offer` for each segment

:::{attention} <font color='#AF8138'>**Bronze Tier with Low or Medium Risk**</font>
:class: dropdown
These customers have a very low CLV, negative expected savings per customer and negative true and predicted savings overall. Retention efforts are expensive and not justified for this group. The objective for these customers should be to maintain baseline engagement, preserve brand relationship and avoid overspending on low-value accounts. A low-cost automated engagement is the most financially rational strategy here.

<u>Recommendation:</u> *Standard Monthly Newsletter + Feature Education ([1](https://thefinancialbrand.com/news/financial-education/is-education-the-new-marketing-how-learning-and-training-unlocks-growth-by-driving-adoption-191119), [2](https://www.gainsight.com/blog/what-is-product-education/))*
:::

:::{attention} <font color='#8B6035'>**Bronze Tier with High Risk**</font>
:class: dropdown
Although these customers are classified as High Risk, they remain low-value customers from a CLV perspective (approximately 109.5 dollars) and this segment generates negative expected savings overall. This indicates that the expected financial benefit of retention of these customers is not high enough to justify expensive interventions. However, because the customers are predicted by the ML model as being highly likely to churn, the bank may still wish to collect customer feedback (e.g. through a customer survey), understand churn drivers and maintain [light-touch engagement](https://pubmed.ncbi.nlm.nih.gov/33276773/) for brand and customer-experience purposes. With this in mind, a good retention strategy would use low-cost, fully automated, highly scalable interventions. A customer satisfaction survey would be particularly useful because it may uncover systemic issues that led to these customers canceling their credit card services, while also keeping the intervention costs minimal.
                                                                                                    <u>Recommendation:</u> *Automated Email + Satisfaction Survey*
:::

:::{attention} <font color='#9EA3AB'>**Silver Tier with Low or Medium Risk**</font>
:class: dropdown
These customers (a) produce positive expected savings and moderate CLV, (b) are valuable enough to justify [lightweight personalization](https://owd.com/blog/types-of-personalization-ecommerce/), [digital incentives](https://www.benamic.com/blog/digital-incentives-replacing-traditional-rewards-2026/) and a moderate level of spending on retention strategy but (c) are not valuable enough to require [labour-intensive interventions like executive calls](https://www.celent.com/en/insights/210203377). So, a good offer to target these at-risk customers would be a balance of scalability (these tiers have approximately 80-120 customers), low operational cost and an easily measurable financial return.

<u>Recommendation:</u> *[Targeted In-App Notification](https://www.adjust.com/glossary/in-app-notifications/) + New Category Cashback Boost*
:::

:::{attention} <font color='#9A938E'>**Silver Tier with High Risk**</font>
:class: dropdown
These are customers with a moderate CLV, very high churn probability and strong positive expected and true returns. Unlike the Bronze customers, these customers generate sufficient expected value to justify meaningful retention incentives. However, the segment size is relatively large (288 customers) the efficiency is low (`expected_savings_per_customer` is only approximately 95.6). So, fully personalized outreach to these customers would likely be operationally expensive and not justified. So, a good strategy would balance personalization, scalability and cost efficiency. An automated email would be a good lightweight option without requiring a hands-on human intervention. If they are offered bonus points on their next credit card transaction then this would also be appropriate because it encourages renewed spending activity on the card, reinforces loyalty program engagement and create a behavioural incentive to continue using the card.

<u>Recommendation:</u> *Automated 'We Miss You' Email + 5k Bonus Points on Next Spend*
:::

:::{attention}  <font color='#D78017'>**Gold Tier with Low or Medium Risk**</font>
:class: dropdown
These customers have a notably higher CLV (nearly triple the CLV of the *Silver* / *Medium* group) and strong positive expected and also true savings. These at-risk customers justify proactive outreach and meaningful incentives but at scalable operational cost. At this level, losing customers becomes financially meaningful. So, stronger retention incentives are economically justified. [Balance transfer offers](https://www.td.com/us/en/personal-banking/learning/borrowing-credit/what-is-a-balance-transfer-credit-card) would be appropriate because they would directly encourage continued credit card usage, increase engagement, and reduce the probability of churning.

<u>Recommendation:</u> *Personalized Email + 0% Balance Transfer Offer for 6 Months*
:::

:::{attention} <font color='#AC661B'>**Gold Tier with High Risk**</font>
:class: dropdown
These customers represent a strategically important segment since they combine high churn probability, high CLV and the strongest and highly *efficient* expected savings per intervention. At this value level, losing these at-risk customers becomes materially expensive. Proactive human intervention would be justified. Unlike the lower value tiers, automated campaigns alone may be insufficient to prevent these customers from churning. So, a good retention strategy would involve direct engagement from the client's management team, meaningful financial incentives and loyalty rewards. For example, an annual fee rebate would help reduce immediate customer dissatisfaction that is leading to their high predicted risk of churn or [price sensitivity](https://www.investopedia.com/terms/p/price-sensitivity.asp). Bonus points would encourage continued product usage and engagement.

<u>Recommendation:</u> *Retention Team Call + 50% Annual Fee Rebate + 10k Points*
:::

:::{attention} <font color='#C8B088'>**Platinum Tier with Low or Medium Risk**</font>
:class: dropdown
These customers are not predicted to have the highest risk of churn but they represent some of the bank's most valuable relationships due to their extremely high CLV (highest and second highest CLV overall) and efficiency (second and third highest `expected_savings_per_customer`). Because these customers are already relatively stable, the objective is not aggressive intervention. Instead, a proactive reinforcement of their relationship with the bank, premium customer experience management and and preserving their long-term loyalty should be prioritized. A good offer should intentionally avoid overly reactive retention tactics, such as fee waivers or urgent targeting calls, which could unnecessarily reduce profitability. A good strategy would emphasize [recognition of the business they have given the bank](https://www.adweek.com/brand-marketing/inside-the-credit-card-wars-for-luxury-customers) and [offering a premium service](https://welcome.comperemedia.com/insights/financial-services/how-premium-credit-card-issuers-are-redefining-strategies-in-2024/). Outreach by one of the relationship managers on the client's management team and luxury benefits would help to strengthen their commitment to the credit card division's premium ecosystem.

<u>Recommendation:</u> *Priority Relationship Manager Check-in + Luxury Lounge Access Upgrade*
:::

:::{attention} <font color='#B69E7B'>**Platinum Tier with High Risk**</font>
:class: dropdown
These customers (a) represent the bank's highest-value and highest-risk customers and (b) generate the second-highest high CLV, strong and efficient expected savings (also the highest). A significant financial loss would be incurred if they churn as predicted by the ML model. This is effectively a [cohort that the client must retain](https://stripe.com/en-ca/resources/more/cohort-analysis-for-businesses). The bank can economically justify [high-touch interventions](https://www.shopify.com/ca/blog/high-touch-customer-service) for this group, including outreach by the client's management team, [premium loyalty incentives](https://r3marketing.ca/wp-content/uploads/2023/11/LoyalT2023_Highlights_WEB.pdf#page=16), and fee waivers. A good retention strategy would prioritize personalization, relationship management, and reinforce their historical commitment to the bank.

<u>Recommendation:</u> *Personalized Executive Call + 1-Year Fee Waiver + 25k Points*
:::

:::{note}
These save offers would have to be offered proactively.
:::

The `get_save_offer()` function implements this business logic. Rather than requiring the client's management team to create targeting strategies for each customer, this approach maps each risk-CLV segment to a standardized intervention policy. For example, a High-Risk Platinum customer receives a high-touch executive retention offer, whereas a Bronze customer receives a low-cost automated engagement campaign. This ensures that retention spending remains proportional to expected customer value (CLV) and estimated savings.

Below is the section of this function that implements the logic to create these offers

In [ ]:
pygments_highlight(
    fpath=str(PROJ_ROOT / "src" / "cc_churn" / "costs_savings.py"),
    unwanted_lines=(
        list(range(0, 10)) + list(range(12, 18)) + list(range(45, 276))
    ),
    style="vs",
)

Next, we'll use the [above table](#overall-segment-attributes) to create a heatmap matrix that shows these aggregated characteristics of each combination of risk level and CLV tier (revenue) and recommended targeting strategy per segment

In [ ]:
# | label: heatmap-all-tiers
text_alt_condition = alt.datum.expected_savings_per_customer > 200
tooltip = [
    alt.Tooltip("num_customers:Q", title="Number of At-Risk Customers"),
    alt.Tooltip("total_intervention_cost:Q", format=",", title="Total Cost"),
    alt.Tooltip(
        "y_pred_proba:Q", format=",.2f", title="Avg. Predicted Probability"
    ),
    alt.Tooltip("clv:Q", format=",.2f", title="Avg. CLV"),
    alt.Tooltip("expected_savings:Q", format=",.2f", title="Expected Savings"),
    alt.Tooltip(
        "savings_error_pct:Q", format=",.2f", title="Savings Error (%)"
    ),
    alt.Tooltip("save_offer:N", title="Recommendation"),
]
ptitle = alt.TitleParams(
    # text="At-Risk Priority Matrix (by CLV Tier)",
    text="At-Risk Customer Outreach and Offer Guide",
    fontSize=18,
    font="Arial",
    anchor="start",
    orient="top",
    dx=80,
    offset=10,
)

chart = vzu.plot_altair_heatmap(
    df_summary_per_clv_tier_risk_level,
    xvar="value_tier:N",
    yvar="risk_level:N",
    textvar="expected_savings_per_customer:Q",
    xsort=["Bronze", "Silver", "Gold", "Platinum"],
    ysort=["High", "Medium", "Low"],
    color_by_col="expected_savings_per_customer:Q",
    legend_title="Savings per Customer",
    border_attrs=dict(stroke="white", strokeWidth=1.0),
    scale_params=dict(scheme="reds"),
    text_fontsize=18,
    text_alt_condition=text_alt_condition,
    tooltip=tooltip,
    ptitle=ptitle,
    xtitle="Revenue Tier (CLV)",
    ytitle="Churn Risk Level",
    fig_size=dict(width=700, height=350),
    save_params=dict(
        fpath=figures_dir / "fig_30_cohort_savings_hmap_all.html"
    ),
)
chart

This heatmap shows how risk of churn intersects with customer lifetime value (revenue) levels. The quantiles for the CLV column ensure the *top value tier* represents a high-priority group of customers.

:::{tip}
The client can hover over any segment in this heatmap and the tooltip for that segment shows exactly what incentive to offer through the *Save Offer* strategy we defined earlier. The tooltips also allow the client to quickly view all the above aggergated characteristics and the *Save Offer*s for all segments.
:::

When comparing the *Platinum / Medium Risk*, *Platinum / Low Risk* and *Gold / High Risk* groups [above](#savings-tradeoff), we are starting to see a tradeoff

1. we can focus on efficiency in which we prioritize the highest `expected_savings_per_customer` and target the *Platinum / Medium Risk* and *Platinum / Low Risk* groups
2. we can focus on maximizing the number of targeted customers and accept a lower efficiency by capturing a lower `expected_savings_per_customer` and target the *Gold / High Risk* which has approximately 2.4 - 2.6 times more customers than either of the other two groups.

Our recommendations have not yet taken available budget into account. This will help determine which area to focus on. So, [as per the project scope](../references/scope/07_reporting_metrics.md#reporting-budget-scenarios), we will now generate recommendations for two budget scenarios - limited budget to target the top *N* customers or unlimited budget so all recommended customers can be targeted.

(budget-scenario-recommendations)=
### Update Lookup Tool Based on Two Budget Scenarios

Depending on the client's budget for retention, there are two possible use-cases for this lookup tool

1. budget of at least approximately 20,000 dollars (nominal)
   - requires at most 400 predicted at-risk customers to be targeted (at an assumed cost of $50 per customer)
2. budget of at least 80,000 dollars
   - allows for up to all 1,600 predicted  at-risk customers to be targeted

Based on the findings from the previous section about the poor performance of the *Bronze* revenue tier, we will exclude tiers with a negative estimated savings before extracting the recommended cohort for both scenarios.

Below is the four-step workflow to get the cohort based on scenario one, in which the client can target at most `N` customers, using the `get_cohort_within_budget()` function

1. First, we'll get top-performing combinations of `risk_level` and `value_tier` in terms of `expected_savings_per_customer` that add up to the required maximum number of customers (`num_customers_max`). Since the budget allows for up to `N` customers to be targeted, `filter_matrix_by_limit()` adds logic to ensure these combinations capture at least `N` customers. These combinations are sorted by expected savings per customer, from highest to lowest.
2. Next, we'll get the customers that meet these top-performing combinations of `risk_level` and `value_tier`, starting from the top ranked combination. This approach exhausts the best-performing combinations entirely before moving down to the next-best one. Note that the customers first are ranked by `expected_savings` to ensure that even within a high-performing group (like *High-Risk* / *Platinum*), we are first picking the best candidates before moving down to the next best combination.
3. Next, `summarize_campaign_mix()` is called to get the breakdown of the buckets (bins) from which these top `N` customers come.
4. Finally, step 4. calculates the estimated total impact (total expected savings) and realizable benefit (average expeced savings in `expected_savings_per_customer`) for the selected cohort.

:::{hint}
These four steps are indicated in the `get_cohort_within_budget()` helper function.
:::

Overall, this logic treats the Value / Risk segments as the primary strategic drivers. It ensures we don't chase *outlier* individual scores at the expense of targeting the most efficient overall categories. Step 2. is the most important and it ensures we are moving from the most profitable segment to our least, until the budget runs out.

#### Scenario 1 - Constrained by Budget of At Least $20,000

Below is the workflow to get the overall cohort for the first scenario in which at most 400 customers can be targeted

In [ ]:
df_ranked_s1, df_campaign_mix_s1, _, _ = costs_sv.get_cohort_within_budget(
    df_business_metrics,
    df_summary_per_clv_tier_risk_level.query("expected_savings > 0"),
    intervention_cost=intervention_cost,
    n=num_customers_max,
)

Below is the updated heatmap for the selected cohort

In [ ]:
text_alt_condition_s1 = alt.datum.expected_savings_per_customer > 400
tooltip_s1 = [
    alt.Tooltip("num_customers:Q", title="Number of At-Risk Customers"),
    alt.Tooltip("total_intervention_cost:Q", format=",", title="Total Cost"),
    alt.Tooltip(
        "y_pred_proba:Q", format=",.2f", title="Avg. Predicted Probability"
    ),
    alt.Tooltip("clv:Q", format=",.2f", title="Avg. CLV"),
    alt.Tooltip("expected_savings:Q", format=",.2f", title="Expected Savings"),
    alt.Tooltip(
        "savings_error_pct:Q", format=",.2f", title="Savings Error (%)"
    ),
    alt.Tooltip("save_offer:N", title="Recommendation"),
]
ptitle_s1 = alt.TitleParams(
    # text="At-Risk Priority Matrix (by CLV Tier)",
    text="At-Risk Cohort Outreach & Offer Guide",
    fontSize=18,
    font="Arial",
    anchor="start",
    orient="top",
    dx=80,
    offset=10,
)

chart_budget = vzu.plot_altair_heatmap(
    df_campaign_mix_s1,
    xvar="value_tier:N",
    yvar="risk_level:N",
    textvar="expected_savings_per_customer:Q",
    xsort=["Bronze", "Silver", "Gold", "Platinum"],
    ysort=["High", "Medium", "Low"],
    color_by_col="expected_savings_per_customer:Q",
    border_attrs=dict(stroke="white", strokeWidth=1.0),
    scale_params=dict(scheme="reds"),
    text_fontsize=18,
    text_alt_condition=text_alt_condition_s1,
    tooltip=tooltip_s1,
    ptitle=ptitle_s1,
    legend_title="Savings per Customer",
    xtitle="Revenue Tier (CLV)",
    ytitle="Churn Risk Level",
    save_params=dict(
        fpath=figures_dir / "fig_31_cohort_savings_hmap_cs1.html"
    ),
    fig_size=dict(width=325, height=350),
)
chart_budget

:::{attention}
Since we are first sorting in reverse order of `expected_savings_per_customer`, only the top two revenue tiers are fully captured. In order to target 400 customers, all three risk levels must be used for these two tiers only, while only the *High Risk* customers from the *Silver* tier must be targeted.
:::

:::{note}
Not all customers within each combination of risk level and value tier are selected for intervention. Instead, the `get_cohort_within_budget()` function prioritizes customers and customer segments with the highest expected savings per customer until the maximum intervention capacity (*N* = 400) is reached. As a result, some lower-priority customers are excluded, even if their segment still has positive expected savings overall. The Platinum segments are fully retained because they have the highest expected savings per customer and therefore rank highest in the prioritization process. However, once the cumulative number of selected customers approaches the intervention limit, only a subset of *Gold* / *Low* group of customers can be included.

**This causes the Gold segment statistics under the budget-constrained strategy to differ from the unconstrained strategy shown in the [first heatmap across all four tiers](#heatmap-all-tiers). The same occurs for the *Silver* / *High* group.**
:::

Below is the section of this function that implements the logic to create these offers

In [ ]:
pygments_highlight(
    fpath=str(PROJ_ROOT / "src" / "cc_churn" / "costs_savings.py"),
    unwanted_lines=(
        list(range(0, 130))
        + list(range(136, 149))
        + list(range(151, 158))
        + list(range(161, 216))
        + list(range(223, 234))
        + list(range(260, 274))
    ),
    style="vs",
)

:::{tip} Explanation of Customer Selection Logic For a Budget, Using Source Code
:class: dropdown
:open: true
The reason why fewer customers are selected can be seen directly in the logic of `get_cohort_within_budget()`.

First, the function ranks all (`risk_level`, `value_tier`) combinations by `expected_savings_per_customer` using

```python
df_matrix_agg.nlargest(n, ["expected_savings_per_customer"])
```

Next, it computes a cumulative customer count

```python
.assign(num_customers_cumsum=lambda df: df["num_customers"].cumsum())
```

The helper function `filter_matrix_by_limit()` then removes any additional segment combinations once the cumulative total exceeds the intervention budget (*N* = 400). Since the Platinum tiers take up a part of the available budget first due to their superior efficiency (`expected_savings_per_customer`), only some of the Gold-tier customers remain eligible for inclusion before reaching the budget limit. So, the final selected cohort contains fewer Gold customers than in the no-budget scenario.

Finally, after the eligible segment combinations are identified, the function applies a second filtering stage:

```python
.nlargest(n, ["expected_savings"])
```

This ranks individual customers within the remaining eligible tiers and retains only the top customers by expected savings. Therefore, the final Gold-tier cohort becomes a more selective, higher-value subset of the original Gold population, which can further modify the aggregated metrics relative to the no-budget results.
:::

#### Scenario 2 - Higher Budget of At Least $80,000

For the second scenario a higher budget is available that allows for targeting all customers predicted to be at risk of churning, so we recommend the client only exclude those in a tier with negative estimated savings.

Below is the workflow to get the overall cohort for this scenario

In [ ]:
df_ranked_s2, df_campaign_mix_s2, _, _ = costs_sv.get_cohort_within_budget(
    df_business_metrics,
    df_summary_per_clv_tier_risk_level.query("expected_savings > 0"),
    intervention_cost=intervention_cost,
    n=len(df_business_metrics),
)

Similar to the first scenario, below is the updated heatmap for the selected cohort

In [ ]:
text_alt_condition_s2 = alt.datum.expected_savings_per_customer > 200
tooltip_s2 = [
    alt.Tooltip("num_customers:Q", title="Number of At-Risk Customers"),
    alt.Tooltip("total_intervention_cost:Q", format=",", title="Total Cost"),
    alt.Tooltip(
        "y_pred_proba:Q", format=",.2f", title="Avg. Predicted Probability"
    ),
    alt.Tooltip("clv:Q", format=",.2f", title="Avg. CLV"),
    alt.Tooltip("expected_savings:Q", format=",.2f", title="Expected Savings"),
    alt.Tooltip(
        "savings_error_pct:Q", format=",.2f", title="Savings Error (%)"
    ),
    alt.Tooltip("save_offer:N", title="Recommendation"),
]
ptitle_s2 = alt.TitleParams(
    # text="At-Risk Priority Matrix (by CLV Tier)",
    text="At-Risk Cohort Outreach & Offer Guide",
    fontSize=18,
    font="Arial",
    anchor="start",
    orient="top",
    dx=85,
    offset=10,
)

chart_budget = vzu.plot_altair_heatmap(
    df_campaign_mix_s2,
    xvar="value_tier:N",
    yvar="risk_level:N",
    textvar="expected_savings_per_customer:Q",
    xsort=["Bronze", "Silver", "Gold", "Platinum"],
    ysort=["High", "Medium", "Low"],
    color_by_col="expected_savings_per_customer:Q",
    legend_title="Savings per Customer",
    border_attrs=dict(stroke="white", strokeWidth=1.0),
    scale_params=dict(scheme="reds"),
    text_fontsize=18,
    text_alt_condition=text_alt_condition_s2,
    tooltip=tooltip_s2,
    ptitle=ptitle_s2,
    xtitle="Revenue Tier (CLV)",
    ytitle="Churn Risk Level",
    fig_size=dict(width=350, height=350),
    save_params=dict(
        fpath=figures_dir / "fig_32_cohort_savings_hmap_cs2.html"
    ),
)
chart_budget

:::{attention}
In order to target all the predicted at-risk customers, three of the four revenue tiers must be used. As we saw from the [earlier heatmap](#heatmap-all-tiers), the expected savings for the fourth tier (*Bronze*) were negative and so this tier is excluded. So, the entire available budget will not be used.
:::

:::{note}
In this higher-budget scenario, the intervention capacity is sufficiently large to allow all economically justifiable at-risk customers to be targeted. As a result, the `get_cohort_within_budget()` function does not exclude customers within the retained (risk_level, value_tier) combinations due to budget limitations. Instead, the only filtering applied is the removal of entire customer segments whose expected savings are negative. So, the `estimated_savings_per_customer` for all risk levels in all tiers are identical in this heatmap to those in the [first heatmap across all three chosen tiers](#heatmap-all-tiers).
:::

:::{tip} Selection Logic For Profitable Customers Under No Budget, From Source Code
:class: dropdown
:open: true
Similar to the previous scenario, the reason why all customers are selected can again be seen in the logic of `get_cohort_within_budget()`.

Specifically, the Bronze tier is excluded using:

```python
.query("expected_savings > 0")
```

because the expected intervention costs for *Bronze* customers exceed the expected recoverable value across all risk levels. Therefore, retaining these customers is not economically justified.

Since no additional customer-level filtering occurs after removing the *Bronze* tier, all remaining *Silver*, *Gold*, and *Platinum* customers are retained in full. Consequently, the `expected_savings_per_customer` values for each risk level and value tier remain identical to those observed in the unconstrained scenario. Unlike the constrained-budget case, no partial segment truncation occurs, so the aggregated campaign metrics are unchanged for the retained tiers.
:::

#### Scenario Comparison

Below we get a summary of the business metrics for these two scenarios

In [ ]:
df_campaign_mixes = (
    pd.concat(
        [
            df_campaign_mix_s1.assign(scenario=1),
            df_campaign_mix_s2.assign(scenario=2),
        ]
    )
    .groupby("scenario", as_index=False)
    .agg(
        {
            "expected_savings": "sum",
            "true_savings": "sum",
            "num_customers": "sum",
            "total_intervention_cost": "sum",
        }
    )
    .assign(
        expected_savings_per_customer=lambda df: df["expected_savings"].div(
            df["num_customers"]
        ),
        savings_error_pct=lambda df: (
            df["expected_savings"]
            .sub(df["true_savings"])
            .div(df["true_savings"])
            .mul(100)
        ),
    )
)

The output is shown below

In [ ]:
gt = (
    GT(df_campaign_mixes)
    .tab_header(md("**Comparison of Performance of Two Budget Scenarios**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["expected_savings"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["savings_error_pct"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["expected_savings_per_customer"]),
    )
    .fmt_number(
        columns=[
            "true_savings",
            "expected_savings",
            "expected_savings_per_customer",
            "savings_error_pct",
        ],
        decimals=1,
    )
    .fmt_number(
        columns=["num_customers", "total_intervention_cost"],
        sep_mark=",",
        decimals=0,
    )
)
gt

**Observations**

1. The errors in the expected savings relative to the true savings (shown in `savings_error_pct`), are similar for both scenarios. We will consider them both negligible as they are both approximately 2%.
2. In terms of expected savings (`expected_savings`), secnario 2 is the best but this comes at the cost of selecting more customers. Correcting for this, by using expected savings per customer (`expected_savings_per_customer`), shows that scenario 1 is the better of the two scenarios.

#### Summary

Scenario 2 targets more customers delivers the higher expected savings per customer. Scenario 1 selects fewer customers (approximately half of scenario 1) and gives the higher average efficiency.

### Append Metadata to Business Metrics per Scenario

In order to prepare for exporting these recommendations, we will now combine the identified cohorts for both scenarios. For convenience, before exporting to disk, we will append a column with a copy of `y_pred` renamed to `is_at_risk` since it indicates if a customer is at-risk (1) or not (0).

This is done below

In [ ]:
df_ranked_s1_s2 = pd.concat(
    [df_ranked_s1.assign(scenario=1), df_ranked_s2.assign(scenario=2)]
).assign(is_at_risk=lambda df: df["y_pred"])

A summary is shown below

In [ ]:
# label: cohort-with-business-metrics
gt = (
    GT(show_df(df_ranked_s1_s2, False)[0].reset_index())
    .tab_header(md("**Summary of Scenario Data**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["num_missing"]),
    )
    .tab_style(
        style=style.fill(color="aliceblue"),
        locations=loc.body(columns=["num_unique"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["dtype"]),
    )
)
gt

## Export Project Deliverables to Private R2 Bucket

Get the current timestamp in the format `YYmmdd_HHMMSS`

In [ ]:
curr_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

Get the name of the sub-directory in the R2 bucket containing the predictions

In [ ]:
sub_directory = r2io.get_latest_s3_file_optimized(
    s3_client, bucket_name, f"{prefix}/", r2_key_pred
).split("/")[1]

Next, export to a file in the R2 bucket with the following file name format `at_risk_customers_with_business_metrics_savings__<best-model-name>__<current-timestamp-YYmmdd_HHMMSS>.parquet.gzip`

In [ ]:
# r2io.export_df_to_r2(
#     s3_client=s3_client,
#     df=df_ranked_s1_s2,
#     bucket_name=bucket_name,
#     r2_key=(
#         f"{prefix}/{sub_directory}/"
#         "at_risk_customers_with_business_metrics_savings__"
#         f"{best_model_name.lower()}__{curr_timestamp}.parquet.gzip"
#     ),
#     verbose=False,
# )

## Conclusion

Our analysis has identified nine profitable customer segments by combining Value Tiers with Risk Levels. We recommend excluding all *Bronze* tier customers from intervention, as these segments are unlikely to yield a positive return.

We evaluated two primary targeting scenarios based on budget availability:

1. Scenario 1 (High Efficiency)
   - By targeting the top 400 customers (requiring a 20,000 dollar budget), we estimate 140,000 dollars in total savings. This approach focuses exclusively on the five highest-performing segments, delivering an average benefit of 357 dollars per customer.
2. Scenario 2 (Maximum Reach)
   - Out of the predicted 1,600 at-risk customers (requiring an 80,000 dollars budget), we estimate the savings to increase to approximately 180,000 dollars. This would be accomplished by targeting the top 850 profitable customers.

Scenario 1 is the superior choice for savings. It captures the bulk of available savings with an approximately 40% higher per-customer efficiency than the full-reach approach.

In order for these estimates to be relevant to other customers, not included in the random sample we used in this project, the other sample must have the same characteristics as those seen in the customers whose data was used in the analysis here. [In a later step](./12_validate_data.ipynb), we develop a data validation model that can be used to validate customer data before being used with the model developed here.